# Checkpoint VI · Examen blanc — notebook de l'examen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_6/02_examen_notebook.ipynb)

Ce notebook accompagne le sujet `01_examen_sujet.md`.

- **Partie A, pendant l'examen** : les deux questions de code, CP6.8 (📦, 2 points : la loss d'un VAE) et CP6.9 (🐛, 1,5 point : un pas de GAN à réparer), avec PyTorch. Écris ton code dans les cellules à compléter, sans `mylearn`, sans les corrigés et sans assistant IA ; `help()` et la documentation officielle de PyTorch sont permis. Rien n'est vérifié pendant l'examen : les cellules « essai » montrent seulement ce que fait ton code. Tes réponses chiffrées et tes explications vont sur ta feuille (`04_mes_reponses.md`).
- **Partie B, après l'examen** : la vérification automatique de ton code et des réponses de ta feuille. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`.

« Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳ (partie A) ou ⏸️ (partie B).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_6/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP6`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · Pendant l'examen : les questions de code

In [ ]:
# Data and tools of the code questions CP6.8 and CP6.9 (run this cell first)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# CP6.8: a batch of 16 MNIST images in [0, 1] (N, 1, 28, 28), the logits a decoder gave for them, and the two heads
# of an encoder with d = 4 latents (mu, log_var); everything in float64
_X8, _ = wb.datasets.load_mnist("train", n=16, normalize=True, seed=68)
X_V = torch.from_numpy(_X8.astype(np.float64)).unsqueeze(1)                        # (16, 1, 28, 28)
_g8 = torch.Generator().manual_seed(68)
LOGITS_V = 2.0 * torch.randn(16, 1, 28, 28, generator=_g8, dtype=torch.float64)  # (16, 1, 28, 28)
MU_V = torch.randn(16, 4, generator=_g8, dtype=torch.float64)                     # (16, 4)
LOGVAR_V = 0.5 * torch.randn(16, 4, generator=_g8, dtype=torch.float64)           # (16, 4)
_ORIGINALS = {"X_V": X_V.clone(), "LOGITS_V": LOGITS_V.clone(), "MU_V": MU_V.clone(), "LOGVAR_V": LOGVAR_V.clone()}

# CP6.9: a 1D GAN; the data follow N(4, 1.25^2)
bce = nn.BCEWithLogitsLoss()                                                      # the mean over the mini-batch


def make_gan(seed=69):
    """(G, D, opt_d, opt_g): a new generator 1-16-1 and discriminator 1-32-32-1 (it gives a logit), Adam(2e-3)."""
    torch.manual_seed(seed)
    G = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))
    D = nn.Sequential(nn.Linear(1, 32), nn.LeakyReLU(0.2), nn.Linear(32, 32), nn.LeakyReLU(0.2), nn.Linear(32, 1))
    opt_d = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.5, 0.999))
    opt_g = torch.optim.Adam(G.parameters(), lr=2e-3, betas=(0.5, 0.999))
    return G, D, opt_d, opt_g


def real_batch(n=64):
    """n samples of the data, N(4, 1.25^2), shape (n, 1)."""
    return 4.0 + 1.25 * torch.randn(n, 1)


print("X_V:", tuple(X_V.shape), "· LOGITS_V:", tuple(LOGITS_V.shape), "· MU_V and LOGVAR_V:", tuple(MU_V.shape),
      "· G parameters:", sum(p.numel() for p in make_gan()[0].parameters()))

### CP6.8 — Écrire la loss d'un VAE en PyTorch et la tester sur un batch fourni 📦 ★★ ⏱️ 15 min · 2 points
**Objectif :** écrire de mémoire la loss d'un VAE avec ses deux termes à la bonne échelle.

Avec PyTorch. La cellule des données fournit un batch de 16 images MNIST `X_V` (valeurs dans $[0, 1]$, forme `(16, 1, 28, 28)`), les **logits** qu'un décodeur a rendus pour elles (`LOGITS_V`, même forme) et les deux têtes d'un encodeur à $d = 4$ latents (`MU_V`, `LOGVAR_V`, forme `(16, 4)`) ; tout est en `float64`.

a) `vae_loss(logits, x, mu, log_var, beta=1.0)` rend le triplet `(loss, rec, kl)` de tenseurs à 0 dimension, **par exemple** (moyennés sur le batch) : `rec`, la cross-entropy binaire entre `x` et `sigmoid(logits)`, **sommée** sur les pixels d'une image ; `kl`, la divergence KL entre $\mathcal{N}(\boldsymbol{\mu}, \operatorname{diag}(e^{\log\boldsymbol{\sigma}^2}))$ et $\mathcal{N}(\mathbf{0}, \mathbf{I})$, sommée sur les latents ; `loss = rec + beta * kl`. Elle marche pour toute forme d'images (`(N, 784)` comme `(N, 1, 28, 28)`) et garde le graphe (le gradient doit pouvoir remonter jusqu'à `logits`, `mu` et `log_var`). **[1,4]**
b) Avec le batch fourni : reporte sur ta feuille $[\text{rec}, \text{kl}]$ (2 décimales). **[0,3]**
c) La loss pour `beta=3` (2 décimales). **[0,3]**

In [ ]:
def vae_loss(logits, x, mu, log_var, beta=1.0):
    """(loss, rec, kl), 0-d tensors per example: rec = BCE summed over the pixels of an image, kl = Gaussian KL summed
    over the latents, both averaged over the batch; loss = rec + beta * kl; CP6.8 a)."""
    raise NotImplementedError("vae_loss")  # TODO CP6.8 a)

In [ ]:
# The tries of part A show what your code does (no check during the exam)
import sys
import traceback


def try_code(question, run):
    """Run one try; a bug shows its traceback without stopping « Run all » (part B must still run)."""
    with wb.attempt(question):
        try:
            run()
        except NotImplementedError:
            raise
        except Exception as exc:
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                       # wb.attempt: « as-tu oublié le return ? »
            traceback.print_exc(file=sys.stdout)
            print(f"❌ {question} : ton code lève l'erreur ci-dessus (la suite du notebook s'exécute quand même).")


def try_8():
    loss, rec, kl = vae_loss(LOGITS_V, X_V, MU_V, LOGVAR_V)
    print("rec =", round(float(rec), 2), "· kl =", round(float(kl), 2), "· loss =", round(float(loss), 2))
    print("loss with beta = 3:", round(float(vae_loss(LOGITS_V, X_V, MU_V, LOGVAR_V, beta=3.0)[0]), 2))


try_code("CP6.8", try_8)

### CP6.9 — Réparer un pas d'entraînement de GAN 🐛 ★ ⏱️ 10 min · 1,5 point
**Objectif :** reconnaître les trois erreurs classiques d'un pas de GAN, et les corriger.

Avec PyTorch. `gan_step_buggy` (cellule suivante) doit faire **un** pas d'entraînement d'un GAN : d'abord le discriminateur (vrais exemples contre faux), puis le générateur (loss non saturante, comme au ch. 27). `make_gan()` rend un générateur, un discriminateur (qui rend un **logit**) et leurs deux optimiseurs ; `real_batch()` tire 64 vrais exemples. Elle contient **trois** lignes fautives ; une erreur peut se déclarer plus loin que sa cause.

a) Écris dans `faulty_9` les lettres des trois lignes **à corriger** (dans n'importe quel ordre). **[0,6]**
b) Écris `gan_step_fixed(G, D, real, opt_d, opt_g)`, la version corrigée : même signature, mêmes sorties, et le même tirage (un seul appel à `torch.randn`, ligne B, avant tout le reste). La cellule d'essai l'enchaîne 1 500 fois (quelques secondes) : la moyenne des exemples générés doit s'approcher de 4. **[0,9]**

In [ ]:
# CP6.9: the step to repair (do not change this cell: copy the function into the next one to fix it)
def gan_step_buggy(G, D, real, opt_d, opt_g):
    """One step of a GAN, the discriminator then the generator; returns (d_loss, g_loss) as floats."""
    ones, zeros = torch.ones(len(real), 1), torch.zeros(len(real), 1)          # A
    fake = G(torch.randn(len(real), 1))                                         # B
    d_loss = bce(D(real), ones) + bce(D(fake), zeros)                           # C
    opt_d.zero_grad()                                                           # D
    d_loss.backward()                                                           # E
    opt_d.step()                                                                # F
    g_loss = bce(D(fake), zeros)                                                # G
    opt_g.zero_grad()                                                           # H
    g_loss.backward()                                                           # I
    opt_d.step()                                                                # J
    return d_loss.item(), g_loss.item()                                         # K

In [ ]:
faulty_9 = ...   # CP6.9 a): the letters of the three faulty lines, e.g. "ABC"


def gan_step_fixed(G, D, real, opt_d, opt_g):
    """gan_step_buggy, fixed (same signature, same returns, one call to torch.randn at line B); CP6.9 b)."""
    raise NotImplementedError("gan_step_fixed")  # TODO CP6.9 b)

In [ ]:
# Try gan_step_fixed: 1 500 steps (a few seconds; no check during the exam)
def train_gan(step, n_steps=1500, seed=69):
    """Train make_gan(seed) with `step` on real_batch(); returns (G, the losses of the last 100 steps)."""
    G, D, opt_d, opt_g = make_gan(seed)
    torch.manual_seed(seed + 1)
    losses = [step(G, D, real_batch(), opt_d, opt_g) for _ in range(n_steps)]
    return G, np.mean(losses[-100:], axis=0)


def try_9():
    G_try, (d_try, g_try) = train_gan(gan_step_fixed)
    with torch.no_grad():
        sample = G_try(torch.randn(2000, 1))
    print(f"generated: mean {sample.mean().item():.2f}, std {sample.std().item():.2f} (data: 4 and 1.25) · "
          f"last 100 steps: d_loss {d_try:.3f}, g_loss {g_try:.3f}")


try_code("CP6.9", try_9)

---

### 🛑 Fin de la partie A

L'examen s'arrête ici. Rends ta copie (enregistre ce notebook et ta feuille), puis, seulement après, passe à la partie B.

## Partie B · Après l'examen : vérification automatique

**Ne commence cette partie qu'une fois ta copie terminée**, examen rendu. Elle vérifie le code de la partie A, puis les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Elle ne note pas : les démarches, les justifications et les questions rédigées se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Exécute les vérifications du code (CP6.8 et CP6.9 ; la seconde entraîne le GAN deux fois, une dizaine de secondes).
3. Reporte ensuite chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). En Python, le séparateur décimal est un **point** (`0.125`), une matrice s'écrit comme une liste de lignes (`[[1, 2], [3, 4]]`), une forme comme une chaîne (`"(2, 5, 7, 7)"`) et les lettres entre guillemets (`"XYZ"`).

Si tu as fermé le notebook depuis l'examen, exécute d'abord la cellule de configuration (tout en haut) et les cellules de la partie A, sauf les essais.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells of part B

import re

from wb.nbtools import StopCheck, as_numpy, letters_text, shape_text, verdict

RESULTS = {}        # sub-ID -> result of wb.check
PROPERTIES = {}     # question -> [(property, passed)], reset each time its check cell runs
PENDING = set()     # the code questions not written yet (⏳)
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans la première cellule de la partie B.")
        _REMINDED.append(True)
        return False
    if "wb" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


def ready(question, *names):
    """True when the cells of part A that define `names` have run (after a kernel restart, run them again)."""
    missing = [name for name in names if name not in globals()]
    if missing:
        verb = "n'existe" if len(missing) == 1 else "n'existent"
        print(f"⚠️ {question} : exécute d'abord les cellules de la partie A ({', '.join(missing)} {verb} pas encore).")
    return not missing


def prop(question, ok, success, failure):
    """One property of the code of part A: ✅ or ❌ (wb.nbtools.verdict), remembered for the summary."""
    PROPERTIES.setdefault(question, []).append((success or failure, bool(ok)))
    verdict(question, ok, success, failure)
    return bool(ok)


def safe(function, *args, **kwargs):
    """(True, result) or (False, the text of the error), or (None, None) while the code is not written:
    a bug of the exam code must not stop part B."""
    try:
        return True, function(*args, **kwargs)
    except NotImplementedError:
        return None, None
    except Exception as exc:
        return False, f"{type(exc).__name__}: {str(exc)[:300]}"


_LIBRARY_ADVICE = " ; si elle fait partie de `mylearn`, ses tests (`pytest`) disent quel cas pose problème."


def code_check(ex_id, value, reminder=""):
    """wb.check of a value computed by your exam function (no advice about mylearn: it is not used here)."""
    result = wb.check(ex_id, value, computed=True, quiet=True)
    message = result.message.replace(_LIBRARY_ADVICE, ".")
    if not result and reminder and not message.startswith("Erreur classique"):
        message += f" Rappel : {reminder}"
    print(f"{'✅' if result else '❌'} Ex {ex_id} : {message}")
    RESULTS[ex_id] = result
    return result


def sorted_letters(value):
    """A set of letters typed in any order ("J, C et G") -> its letters in alphabetical order ("CGJ")."""
    return letters_text(value, sort=True)


def restore_data(question, names):
    """Put back the data of part A if a try of part A changed them in place (a function that writes into its input)."""
    changed = [name for name in names
               if name not in globals() or not torch.equal(globals()[name], _ORIGINALS[name])]
    for name in changed:
        globals()[name] = _ORIGINALS[name].clone()
    if changed:
        print(f"⚠️ {question} : {', '.join(changed)} avai{'en' if len(changed) > 1 else ''}t changé depuis la cellule des "
              f"données (une fonction de la partie A écrit-elle dans ce qu'elle reçoit ?) : la vérification repart "
              f"des données d'origine.")

### Le code de la partie A

In [ ]:
# Checks of vae_loss (CP6.8), against the formulas of chapter 25
if exam_over() and ready("CP6.8", "np", "torch", "F", "X_V", "LOGITS_V", "MU_V", "LOGVAR_V", "_ORIGINALS", "vae_loss"):
    PROPERTIES["CP6.8"] = []
    PENDING.discard("CP6.8")
    for key in ("CP6.8b", "CP6.8c"):
        RESULTS.pop(key, None)
    restore_data("CP6.8", ["X_V", "LOGITS_V", "MU_V", "LOGVAR_V"])

    def oracle_8(logits, x, mu, log_var, beta=1.0):
        """(loss, rec, kl) as floats: BCE summed over the pixels and Gaussian KL summed over the latents, per example."""
        n = x.shape[0]
        rec = float(F.binary_cross_entropy_with_logits(logits, x, reduction="sum")) / n
        kl = float(0.5 * (mu ** 2 + log_var.exp() - 1 - log_var).sum()) / n
        return rec + beta * kl, rec, kl

    def as_float(value):
        try:
            return float(value)
        except (TypeError, ValueError, RuntimeError):
            return None

    inputs = [t.clone() for t in (LOGITS_V, X_V, MU_V, LOGVAR_V)]
    ok, result = safe(vae_loss, *inputs)
    if ok is None:
        print("⏳ Ex CP6.8 : vae_loss n'est pas encore écrite.")
        PENDING.add("CP6.8")
    elif not ok:
        prop("CP6.8", False, "", f"vae_loss lève une erreur : {result}")
    elif not isinstance(result, (tuple, list)) or len(result) != 3:
        prop("CP6.8", False, "", f"vae_loss doit rendre le triplet (loss, rec, kl) ; reçu {type(result).__name__}"
             + (f" de longueur {len(result)}" if isinstance(result, (tuple, list)) else "") + ".")
    else:
        got = [as_float(v) for v in result]
        want = oracle_8(LOGITS_V, X_V, MU_V, LOGVAR_V)
        prop("CP6.8", all(isinstance(v, torch.Tensor) and v.dim() == 0 for v in result),
             "loss, rec et kl sont des tenseurs à 0 dimension.",
             "loss, rec et kl doivent être des tenseurs à 0 dimension (pas des float : le gradient doit pouvoir "
             "remonter ; pas un tenseur par image : la moyenne sur le batch).")
        if None not in got:
            prop("CP6.8", abs(got[1] - want[1]) < 1e-6 * abs(want[1]),
                 "rec : la cross-entropy binaire sur les logits, sommée sur les pixels, moyennée sur les images.",
                 f"rec vaut {got[1]:.4f}, attendu {want[1]:.4f} : binary_cross_entropy_with_logits (pas de sigmoïde "
                 "avant), une SOMME sur les pixels de chaque image, divisée par le nombre d'images.")
            prop("CP6.8", abs(got[2] - want[2]) < 1e-9 + 1e-6 * abs(want[2]),
                 "kl : ½(μ² + σ² − 1 − ln σ²) sommée sur les latents, moyennée sur les images.",
                 f"kl vaut {got[2]:.4f}, attendu {want[2]:.4f} : ½(μ² + exp(log_var) − 1 − log_var), sommée sur les "
                 "latents, divisée par le nombre d'images.")
            prop("CP6.8", abs(got[0] - want[0]) < 1e-6 * abs(want[0]), "loss = rec + kl avec beta = 1.",
                 f"loss vaut {got[0]:.4f}, attendu {want[0]:.4f} = rec + beta * kl.")
        prop("CP6.8", all(torch.equal(a, b) for a, b in zip(inputs, (LOGITS_V, X_V, MU_V, LOGVAR_V))),
             "vae_loss ne modifie pas ses entrées.", "vae_loss modifie un tenseur qu'elle reçoit : elle doit le laisser intact.")
        ok4, res4 = safe(vae_loss, LOGITS_V, X_V, MU_V, LOGVAR_V, beta=3.0)
        want4 = oracle_8(LOGITS_V, X_V, MU_V, LOGVAR_V, beta=3.0)[0]
        got4 = as_float(res4[0]) if ok4 and isinstance(res4, (tuple, list)) and len(res4) == 3 else None
        prop("CP6.8", got4 is not None and abs(got4 - want4) < 1e-6 * abs(want4),
             "beta multiplie la KL (beta = 3).", f"avec beta = 3, attendu loss = {want4:.4f}, reçu {got4} : "
             "beta multiplie la KL seulement, et vient de l'argument.")
        # another shape (N, 784), other sizes, and the gradient
        g_o = torch.Generator().manual_seed(681)
        lo = torch.randn(3, 784, generator=g_o, dtype=torch.float64, requires_grad=True)
        xo = torch.rand(3, 784, generator=g_o, dtype=torch.float64)
        mo = torch.randn(3, 5, generator=g_o, dtype=torch.float64, requires_grad=True)
        vo = torch.randn(3, 5, generator=g_o, dtype=torch.float64, requires_grad=True)
        ok_o, res_o = safe(vae_loss, lo, xo, mo, vo, beta=0.5)
        want_o = oracle_8(lo.detach(), xo, mo.detach(), vo.detach(), beta=0.5)
        try:
            same_o = bool(ok_o and abs(float(res_o[0]) - want_o[0]) < 1e-6 * abs(want_o[0]))
        except (TypeError, ValueError, IndexError, RuntimeError):
            same_o = False
        prop("CP6.8", same_o, "avec des images aplaties (3, 784), 5 latents et beta = 0,5, la loss du contrat.",
             "avec des images aplaties (3, 784), 5 latents et beta = 0,5, la loss n'est pas celle du contrat : utilise "
             "les tailles des arguments (pas 16 images, 784 pixels ou 4 latents écrits en dur).")
        try:
            res_o[0].backward()
            flows = all(t.grad is not None and float(t.grad.abs().sum()) > 0 for t in (lo, mo, vo))
        except Exception:
            flows = False
        prop("CP6.8", flows, "le gradient remonte jusqu'aux logits, à mu et à log_var.",
             "loss.backward() ne donne pas de gradient à logits, mu et log_var : pas de .item(), de .detach() ni de "
             "torch.no_grad() dans vae_loss.")
        if None not in got:
            code_check("CP6.8b", [got[1], got[2]], "[rec, kl] avec le batch fourni et beta = 1.")
        if got4 is not None:
            code_check("CP6.8c", got4, "la loss avec beta = 3.")

In [ ]:
# Checks of gan_step_fixed (CP6.9), against the step of chapter 27, weight by weight (about ten seconds)
if exam_over() and ready("CP6.9", "torch", "nn", "bce", "make_gan", "real_batch", "gan_step_fixed", "faulty_9"):
    PROPERTIES["CP6.9"] = []
    PENDING.discard("CP6.9")
    RESULTS.pop("CP6.9a", None)
    if faulty_9 is ...:
        print("⏳ Ex CP6.9a : faulty_9 n'est pas encore rempli.")
    else:
        RESULTS["CP6.9a"] = wb.check("CP6.9a", sorted_letters(faulty_9))

    def reference_step(G, D, real, opt_d, opt_g):
        """The GAN step of chapter 27: D on real and detached fakes, then G with the label 1 (non-saturating)."""
        ones, zeros = torch.ones(len(real), 1), torch.zeros(len(real), 1)
        fake = G(torch.randn(len(real), 1))
        d_loss = bce(D(real), ones) + bce(D(fake.detach()), zeros)
        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()
        g_loss = bce(D(fake), ones)
        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()
        return d_loss.item(), g_loss.item()

    def run_steps(step, n=3, seed=69):
        """(ok, losses or error, G weights, D weights) after n steps from make_gan(seed)."""
        G, D, opt_d, opt_g = make_gan(seed)
        torch.manual_seed(seed + 1)
        losses = []
        for _ in range(n):
            ok, out = safe(step, G, D, real_batch(), opt_d, opt_g)
            if not ok:
                return ok, out, None, None
            losses.append(out)
        return True, losses, [p.detach().clone() for p in G.parameters()], [p.detach().clone() for p in D.parameters()]

    ok, mine, mine_g, mine_d = run_steps(gan_step_fixed)
    if ok is None:
        print("⏳ Ex CP6.9 : gan_step_fixed n'est pas encore écrite.")
        PENDING.add("CP6.9")
    elif not ok:
        prop("CP6.9", False, "", f"gan_step_fixed lève une erreur : {mine}" + (
            " (un backward à travers un graphe déjà libéré : le pas de D a-t-il détaché les faux ?)"
            if "backward through the graph" in str(mine) else ""))
    else:
        _, ref, ref_g, ref_d = run_steps(reference_step)
        try:
            pairs_ok = all(len(m) == 2 and isinstance(m[0], float) and isinstance(m[1], float) for m in mine)
        except TypeError:
            pairs_ok = False
        prop("CP6.9", pairs_ok, "gan_step_fixed rend le couple (d_loss, g_loss) de float.",
             "gan_step_fixed doit rendre le couple (d_loss, g_loss), deux float Python (.item()).")
        same_d = all(torch.allclose(a, b, rtol=1e-5, atol=1e-7) for a, b in zip(mine_d, ref_d))
        same_g = all(torch.allclose(a, b, rtol=1e-5, atol=1e-7) for a, b in zip(mine_g, ref_g))
        prop("CP6.9", same_d, "après trois pas, les poids du discriminateur sont ceux du pas de référence.",
             "après trois pas, les poids du discriminateur diffèrent de ceux du pas de référence : vrais exemples avec "
             "le label 1, faux DÉTACHÉS avec le label 0, et un seul tirage torch.randn (ligne B).")
        prop("CP6.9", same_g, "après trois pas, les poids du générateur sont ceux du pas de référence.",
             "après trois pas, les poids du générateur diffèrent : sa loss donne le label 1 aux faux (non saturante), "
             "et c'est opt_g qui fait le pas.")
        if pairs_ok and len(mine) == len(ref):
            prop("CP6.9", np.allclose(np.array(mine, dtype=float), np.array(ref, dtype=float), rtol=1e-5),
                 "les losses rendues sont celles du pas de référence.",
                 "les losses rendues diffèrent de celles du pas de référence : d_loss = BCE(D(vrais), 1) + BCE(D(faux), 0), "
                 "g_loss = BCE(D(faux), 1).")
        if same_d and same_g:
            G_c, (d_c, g_c) = train_gan(gan_step_fixed)
            with torch.no_grad():
                sample_c = G_c(torch.randn(2000, 1, generator=torch.Generator().manual_seed(9)))
            prop("CP6.9", abs(sample_c.mean().item() - 4.0) < 0.5,
                 f"après 1 500 pas, les exemples générés ont une moyenne de {sample_c.mean().item():.2f} (les données : 4).",
                 f"après 1 500 pas, la moyenne générée vaut {sample_c.mean().item():.2f}, loin de 4.")

### Les réponses de ta feuille

**CP6.1 — Questions flash : dix réponses courtes sur la partie VI**

In [ ]:
# CP6.1: the values written on your answer sheet
answer_CP6_1a = ...  # a) a number, 1 decimal
answer_CP6_1b = ...  # b) True or False
answer_CP6_1c = ...  # c) a letter, e.g. "B"
answer_CP6_1d = ...  # d) a number, 2 decimals
answer_CP6_1e = ...  # e) a shape like "(1, 2)"
answer_CP6_1f = ...  # f) a number
answer_CP6_1g = ...  # g) True or False
answer_CP6_1h = ...  # h) a shape like "(1, 2)"
answer_CP6_1i = ...  # i) a whole number
answer_CP6_1j = ...  # j) True or False

if exam_over():
    for letter, answer in zip("abcdefghij", [answer_CP6_1a, answer_CP6_1b, answer_CP6_1c, answer_CP6_1d, shape_text(answer_CP6_1e), answer_CP6_1f, answer_CP6_1g, shape_text(answer_CP6_1h), answer_CP6_1i, answer_CP6_1j]):
        RESULTS[f"CP6.1{letter}"] = wb.check(f"CP6.1{letter}", answer)

**CP6.2 — VAE : un tirage reparamétrisé et sa divergence KL**

In [ ]:
# CP6.2: the values written on your answer sheet
answer_CP6_2a = ...  # a) [z1, z2], 4 decimals
answer_CP6_2b = ...  # b) [KL1, KL2], 4 decimals
answer_CP6_2c = ...  # c) a number, 4 decimals
answer_CP6_2d = ...  # d) a number, 4 decimals

if exam_over():
    for letter, answer in zip("abcd", [answer_CP6_2a, answer_CP6_2b, answer_CP6_2c, answer_CP6_2d]):
        RESULTS[f"CP6.2{letter}"] = wb.check(f"CP6.2{letter}", answer)

**CP6.3 — Deux transitions : mise à jour Q-learning puis SARSA**

In [ ]:
# CP6.3: the values written on your answer sheet
answer_CP6_3a = ...  # a) a number, 4 decimals
answer_CP6_3b = ...  # b) a number, 4 decimals
answer_CP6_3c = ...  # c) a number, 4 decimals
answer_CP6_3d = ...  # d) a probability, 4 decimals

if exam_over():
    for letter, answer in zip("abcd", [answer_CP6_3a, answer_CP6_3b, answer_CP6_3c, answer_CP6_3d]):
        RESULTS[f"CP6.3{letter}"] = wb.check(f"CP6.3{letter}", answer)

**CP6.4 — Discriminateur optimal et losses à l'équilibre**

In [ ]:
# CP6.4: the values written on your answer sheet
answer_CP6_4b = ...  # b) [loss of D, loss of G], 3 decimals
answer_CP6_4c = ...  # c) a number, 4 decimals

if exam_over():
    for letter, answer in zip("bc", [answer_CP6_4b, answer_CP6_4c]):
        RESULTS[f"CP6.4{letter}"] = wb.check(f"CP6.4{letter}", answer)

**CP6.5 — Matrice de Gram de trois cartes d'activation**

In [ ]:
# CP6.5: the values written on your answer sheet
answer_CP6_5a = ...  # a) F F^T as a list of rows
answer_CP6_5b = ...  # b) [G00, G02], 4 decimals

if exam_over():
    for letter, answer in zip("ab", [answer_CP6_5a, answer_CP6_5b]):
        RESULTS[f"CP6.5{letter}"] = wb.check(f"CP6.5{letter}", answer)

**CP6.6 — Prédire : γ → 0 puis ε = 0 pour un agent Q-learning sur Flippers**

In [ ]:
# CP6.6: the values written on your answer sheet
answer_CP6_6a = ...  # a) a letter, e.g. "B"
answer_CP6_6b = ...  # b) a letter, e.g. "B"

if exam_over():
    for letter, answer in zip("ab", [answer_CP6_6a, answer_CP6_6b]):
        RESULTS[f"CP6.6{letter}"] = wb.check(f"CP6.6{letter}", answer)

**CP6.7 — Trois courbes, trois diagnostics : VAE, GAN, apprentissage par renforcement**

In [ ]:
# CP6.7: the values written on your answer sheet
answer_CP6_7a = ...  # a) a letter A to F
answer_CP6_7b = ...  # b) a letter A to F
answer_CP6_7c = ...  # c) a letter A to F

if exam_over():
    for letter, answer in zip("abc", [letters_text(answer_CP6_7a), letters_text(answer_CP6_7b), letters_text(answer_CP6_7c)]):
        RESULTS[f"CP6.7{letter}"] = wb.check(f"CP6.7{letter}", answer)

**CP6.10 — Table Q ou réseau : estimer la mémoire nécessaire**

In [ ]:
# CP6.10: the values written on your answer sheet
answer_CP6_10a = ...  # a) megabytes, 2 decimals
answer_CP6_10b = ...  # b) megabytes, 2 decimals
answer_CP6_10c = ...  # c) gigabytes, 2 decimals
answer_CP6_10d = ...  # d) a whole number

if exam_over():
    for letter, answer in zip("abcd", [answer_CP6_10a, answer_CP6_10b, answer_CP6_10c, answer_CP6_10d]):
        RESULTS[f"CP6.10{letter}"] = wb.check(f"CP6.10{letter}", answer)

**CP6.13 — Retour sur les parties IV et V : convolution et pas d'Adam**

In [ ]:
# CP6.13: the values written on your answer sheet
answer_CP6_13a = ...  # a) a shape like "(1, 2, 3, 4)"
answer_CP6_13b = ...  # b) a whole number
answer_CP6_13c = ...  # c) a shape like "(1, 2, 3, 4)"
answer_CP6_13d = ...  # d) w1, 4 decimals
answer_CP6_13e = ...  # e) w2, 4 decimals

if exam_over():
    for letter, answer in zip("abcde", [shape_text(answer_CP6_13a), answer_CP6_13b, shape_text(answer_CP6_13c), answer_CP6_13d, answer_CP6_13e]):
        RESULTS[f"CP6.13{letter}"] = wb.check(f"CP6.13{letter}", answer)

### Bilan

In [ ]:
# Summary of the automatic checks
import re


def natural(sub_id):
    """CP6.2a before CP6.10a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP6\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    for question in ("CP6.8", "CP6.9"):
        checked_props = [ok for _, ok in PROPERTIES.get(question, [])]
        state = " (pas encore écrite : ⏳)" if question in PENDING else ""
        print(f"Propriétés du code de {question} : {sum(checked_props)} sur {len(checked_props)}{state}.")
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; les explications (CP6.2 e, CP6.3 e, CP6.4 a, d et e, CP6.5 c, CP6.6 c, CP6.10 e, CP6.11, CP6.12, CP6.13 f) ; et, pour CP6.8, les nombres recopiés sur ta feuille (seule ta fonction est vérifiée). Tout cela se corrige avec `03_examen_corrige.md`. Ensuite : la remédiation du corrigé (relis la synthèse, `05_synthese.md`, pour les notions ratées), puis le mini-projet.